# 09 · Single-Source Baselines (RQ1)

Trains the full method on each source dataset independently and evaluates on the
external Mendeley set. Comparison against the multi-source result isolates the
contribution of source diversity relative to algorithmic components, addressing
RQ1. Single-source training uses one domain, so the domain-adversarial component
is disabled.

## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest = T.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Copied array in 36s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


## Configuration check

In [3]:
print('FULL_METHOD:', config.FULL_METHOD_NAME)
print('Components:', config.FULL_METHOD)

FULL_METHOD: H_combined
Components: {'sampler': True, 'noise_loss': True, 'curriculum': True, 'domain_adv': True, 'hierarchical': True, 'ordinal': True, 'use_quality': True}


## Execution

The full method (without the domain-adversarial component) is trained on each source dataset alone and evaluated on the held-out Mendeley set.

In [4]:
mech = dict(config.FULL_METHOD)
mech['domain_adv'] = False
test_ds = 'mendeley'
results = []
for src in ['oai','nhanes3','mrkr']:
    run = f'single_{src}_seed0'
    pool = manifest[manifest['dataset']==src]
    va = pool.sample(frac=0.15, random_state=0)
    tr = pool.drop(va.index)
    te = manifest[manifest['dataset']==test_ds]
    print(f'--- {run}: train {src} ({len(tr):,}), test {test_ds} ({len(te):,}) ---', flush=True)
    r = T.run_training(run, tr.reset_index(drop=True), va.reset_index(drop=True),
                       te.reset_index(drop=True), mech, 0,
                       config.CKPT_DIR, config.RESULTS_DIR, log_fn=print)
    results.append(r)

df = pd.DataFrame(results)
print('\n=== Single-source baselines (test on Mendeley) ===')
print(df[['run_name','internal_acc5','external_acc5','external_qwk','gap']].to_string(index=False))
df.to_csv(str(config.RESULTS_DIR/'single_source_baselines.csv'), index=False)

--- single_oai_seed0: train oai (7,265), test mendeley (8,259) ---
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 238MB/s]


  [single_oai_seed0] resume ep8
  [single_oai_seed0] ep9/18 loss=2.645 tr=0.483 val=0.502 gap=-0.018 qwk=0.470 grl=0.00 (34s)
  [single_oai_seed0] ep10/18 loss=2.618 tr=0.475 val=0.505 gap=-0.030 qwk=0.456 grl=0.00 (17s)
  [single_oai_seed0] ep11/18 loss=2.564 tr=0.490 val=0.512 gap=-0.022 qwk=0.503 grl=0.00 (16s)
  [single_oai_seed0] ep12/18 loss=2.579 tr=0.492 val=0.512 gap=-0.020 qwk=0.483 grl=0.00 (17s)
  [single_oai_seed0] ep13/18 loss=2.551 tr=0.491 val=0.511 gap=-0.020 qwk=0.509 grl=0.00 (16s)
  [single_oai_seed0] ep14/18 loss=2.569 tr=0.486 val=0.516 gap=-0.029 qwk=0.498 grl=0.00 (16s)
  [single_oai_seed0] ep15/18 loss=2.549 tr=0.490 val=0.516 gap=-0.026 qwk=0.504 grl=0.00 (17s)
  [single_oai_seed0] ep16/18 loss=2.515 tr=0.495 val=0.519 gap=-0.024 qwk=0.496 grl=0.00 (16s)
  [single_oai_seed0] ep17/18 loss=2.598 tr=0.486 val=0.517 gap=-0.031 qwk=0.495 grl=0.00 (16s)
  [single_oai_seed0] early stop ep17
  [single_oai_seed0] DONE int=0.519 ext=0.305 gap=0.214 qwk=0.354
--- single_

## Comparison with multi-source

The best single-source external accuracy is compared against the multi-source full-method result to quantify the effect of source diversity.

In [5]:
best_single = df['external_acc5'].max()
best_src = df.loc[df['external_acc5'].idxmax(),'run_name']
print(f'Best single-source external: {best_single:.3f} ({best_src})')

try:
    abl = pd.read_csv(str(config.RESULTS_DIR/'ablation_summary.csv'))
    h = abl[abl['run_name'].str.contains('H_combined')]
    if len(h):
        multi = float(h['external_acc5'].iloc[0])
        print(f'Multi-source full method (H): {multi:.3f}')
        print(f'Diversity effect: {multi-best_single:+.3f} external accuracy')
        print('Multi-source > best single-source confirms source diversity helps (RQ1).' if multi>best_single
              else 'Best single source matches multi-source — that source is representative.')
except Exception:
    print('Run notebook 07 (ablation) to obtain the multi-source comparison point.')

Best single-source external: 0.305 (single_oai_seed0)
Multi-source full method (H): 0.375
Diversity effect: +0.070 external accuracy
Multi-source > best single-source confirms source diversity helps (RQ1).
